In [0]:
# ── CELL 1: Load Secrets & Configure ADLS ────────────────────
connection_string = dbutils.secrets.get(
    scope="clickstream-scope",
    key="eventhub-connection-string"
)
storage_account = dbutils.secrets.get(
    scope="clickstream-scope",
    key="storage-account-name"
)
storage_key = dbutils.secrets.get(
    scope="clickstream-scope",
    key="storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

EVENTHUB_NAME = "clickstream-events"
EVENTHUB_NAMESPACE = "evhbns-clickstream-dev"

print("Secrets loaded successfully.")

Secrets loaded successfully.


In [0]:
# ── CELL 2: Read Raw Stream from Event Hubs ──────────────────
kafka_options = {
    "kafka.bootstrap.servers": f"{EVENTHUB_NAMESPACE}.servicebus.windows.net:9093",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{connection_string}";',
    "subscribe": EVENTHUB_NAME,
    "startingOffsets": "latest",
    "kafka.request.timeout.ms": "60000",
    "kafka.session.timeout.ms": "30000",
    "failOnDataLoss": "false"
}

raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

print("Raw stream schema:")
raw_stream.printSchema()

Raw stream schema:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [0]:
# ── CELL 3: Parse & Enrich Stream ────────────────────────────
from pyspark.sql.functions import (
    col, from_json, to_timestamp, hour, dayofweek,
    when, lit, current_timestamp
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

clickstream_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("page_url", StringType(), True),
    StructField("referrer_url", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("browser", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("time_spent_seconds", IntegerType(), True)
])

parsed_stream = (
    raw_stream
    .select(
        from_json(
            col("value").cast("string"),
            clickstream_schema
        ).alias("data"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    )
    .select("data.*", "partition", "offset", "kafka_timestamp")
)

enriched_stream = (
    parsed_stream
    .withColumn("event_timestamp", to_timestamp(col("timestamp")))
    .withColumn("hour_of_day", hour(col("event_timestamp")))
    .withColumn("day_of_week", dayofweek(col("event_timestamp")))
    .withColumn(
        "is_product_page",
        when(col("page_url").contains("/products"), lit(True))
        .otherwise(lit(False))
    )
    .withColumn(
        "is_conversion",
        when(col("event_type") == "purchase", lit(True))
        .otherwise(lit(False))
    )
    .withColumn("ingested_at", current_timestamp())
)

print("Enriched stream schema:")
enriched_stream.printSchema()

Enriched stream schema:
root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- referrer_url: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- browser: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- time_spent_seconds: integer (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_product_page: boolean (nullable = false)
 |-- is_conversion: boolean (nullable = false)
 |-- ingested_at: times

In [0]:
# ── CELL 4: Write Stream to ADLS Gen2 ────────────────────────
CONTAINER = "raw"
CHECKPOINT_PATH = (
    f"abfss://{CONTAINER}@{storage_account}"
    f".dfs.core.windows.net/checkpoints/clickstream"
)
OUTPUT_PATH = (
    f"abfss://{CONTAINER}@{storage_account}"
    f".dfs.core.windows.net/clickstream_events"
)

query = (
    enriched_stream
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("path", OUTPUT_PATH)
    .partitionBy("day_of_week", "hour_of_day")
    .trigger(processingTime="30 seconds")
    .start()
)

print(f"Streaming query started: {query.id}")
print(f"Writing to: {OUTPUT_PATH}")
print("Status:", query.status)

Streaming query started: fdd83fa6-b1c8-4091-a76b-be6f247ede04
Writing to: abfss://raw@[REDACTED].dfs.core.windows.net/clickstream_events
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
